In [16]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
import importlib
import yaml
import torch

In [2]:
## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/pilot_ssl_word_resnet50.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 4 
config['hparas']['batch_size'] = 128

In [3]:
config

{'data': {'root': '/mnt/ceph/users/jfeather/data/training_datasets_audio/JSIN_all_v3/subsets/'},
 'audio_rep': {'name': 'cochleagram_1', 'on_gpu': True},
 'audio_transforms': {'low_snr': -10, 'high_snr': 10, 'dbspl': 60},
 'val_metric': 'val_ppe',
 'val_metric_mode': 'min',
 'hparas': {'epochs': 10,
  'global_batch_size': 768,
  'optimizer': 'LARS',
  'lr': 0.2,
  'num_warmup_steps_or_ratio': 0.1,
  'lambda_ssl': 1,
  'valid_step': 5000,
  'ssl_task': 'word',
  'ssl_loss_str': 'mmcr',
  'ssl_loss': 'MMCR_Loss',
  'batch_size': 128},
 'model': {'arch_name': 'SSLAudioModel',
  'arch_kwargs': {'projector_dims': [512, 512],
   'proj_out_dim': 2048,
   'n_classes': 794,
   'supervised': False}},
 'num_workers': 4}

### Test batched dataloaer 

In [23]:
from robustness.audio_functions.audio_transforms import *

transform = AudioCompose(
    [
        AudioToTensor(),
        CombineWithRandomDBSNR()
    ]
)

In [59]:
### make sure dataloader workds 
import lightning_scripts.jsinV3DataLoader_precombined_batched as jsin_batched 
import robustness.audio_functions.audio_transforms as at 

importlib.reload(jsin_batched)

jsinV3_precombined_paired_batched = jsin_batched.jsinV3_precombined_paired_batched

transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.CombineWithRandomDBSNR(low_snr=config['audio_transforms']['low_snr'],
                                          high_snr=config['audio_transforms']['high_snr']),
                at.DBSPLNormalizeForegroundAndBackground(dbspl=config['audio_transforms']['dbspl']),
                at.UnsqueezeAudio(dim=0) # dim=0 here so batches of audio from dataloader will be (Batch, 1, Time)
            ])


def collate_fn(batch):
    """
    Logic for generating views of speech and noise egs. 
    """
    batch = batch[0] # unbox wrapper added by dataloader 
    signal_11, signal_12, signal_21, signal_22 = [], [], [], []
    target_1 = batch[-2] # labels already collated 
    target_2 = batch[-1] # labels already collated 
    # convert labels to torch tensors 
    if isinstance(target_1, dict):
        for task_key, task_labels in target_1.items():
            target_1[task_key] = torch.from_numpy(task_labels)
    if isinstance(target_2, dict):
        for task_key, task_labels in target_2.items():
            target_2[task_key] = torch.from_numpy(task_labels)
    else:
        target_1 = torch.from_numpy(target_1) 
        target_2 = torch.from_numpy(target_2) 
    # convert signal and noise into signal
    for (signal_1, signal_2, noise_1, noise_2) in  zip(*batch[:4]):
        sig_11, _ = transforms(signal_1, noise_1)
        sig_12, _ = transforms(signal_1, noise_2)
        sig_21, _ = transforms(signal_2, noise_1)
        sig_22, _ = transforms(signal_2, noise_2)
        # dummy handle noise-only signals:
        sig_11 = sig_12 if sig_11 is None else sig_11
        sig_12 = sig_11 if sig_12 is None else sig_12
        sig_21 = sig_22 if sig_21 is None else sig_21
        sig_22 = sig_21 if sig_22 is None else sig_22

        signal_11.append(sig_11)
        signal_12.append(sig_12)
        signal_21.append(sig_21)
        signal_22.append(sig_22)

    signal_11 = torch.cat(signal_11).unsqueeze(1) # add back channel dim
    signal_12 = torch.cat(signal_12).unsqueeze(1) # add back channel dim
    signal_21 = torch.cat(signal_21).unsqueeze(1) # add back channel dim
    signal_22 = torch.cat(signal_22).unsqueeze(1) # add back channel dim

    return signal_11, signal_12, signal_21, signal_22, target_1, target_2



train_dset = jsinV3_precombined_paired_batched(root=config['data']['root'], train=True, transform=None, batch_size=64)
loader_batched = torch.utils.data.DataLoader(
            train_dset,
            batch_size=1,
            num_workers=0, 
            pin_memory=True,
            # persistent_workers=True,
            collate_fn=collate_fn,
            shuffle=False,
        )


In [32]:
# class_map = train_dset.class_map()

In [41]:
batched_eg = next(iter(loader_batched))

In [42]:
sig11, sig12, sig21, sig22, label1, label2 = batched_eg

In [43]:
import IPython.display as ipd

In [44]:
label1[0], label1[2]

(tensor(134), tensor(618))

In [52]:
Audio(sig22[20].squeeze(), rate=20_000)

In [65]:
%%timeit
total = 100
for batch in loader_batched:
    if total == 0:
        break
    total -= 1 

5.53 s ± 80.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Compare to unbatched version

In [4]:
import robustness.audio_functions.jsinV3DataLoader_precombined as jsinV3DataLoader_precombined
import robustness.audio_functions.audio_transforms as at 

importlib.reload(jsinV3DataLoader_precombined)

jsinV3_precombined_paired = jsinV3DataLoader_precombined.jsinV3_precombined_paired
train_dset = jsinV3_precombined_paired(root=config['data']['root'], train=True, transform=transforms,)
loader = torch.utils.data.DataLoader(
            train_dset,
            batch_size=64,
            num_workers=0, 
            pin_memory=True,
            # persistent_workers=True,
            shuffle=False,
        )


NameError: name 'transforms' is not defined

In [39]:
train_dset[843]

(tensor([[ 0.0070,  0.0025, -0.0099,  ...,  0.0144,  0.0150, -0.0089]]),
 tensor([[-2.5322e-02, -3.3431e-05, -2.8150e-02,  ...,  7.5955e-03,
           3.0119e-02, -3.1161e-02]]),
 tensor([[ 0.0302,  0.0242,  0.0038,  ..., -0.0347, -0.0237, -0.0353]]),
 tensor([[-0.0266,  0.0303, -0.0314,  ..., -0.0666, -0.0031, -0.0957]]),
 np.int64(644),
 np.int64(484))

In [64]:
%%timeit

total = 100
for batch in loader:
    if total == 0:
        break
    total -= 1 

24 s ± 69.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
sig11, sig12, sig21, sig22, label1, label2 = batch

NameError: name 'batch' is not defined

In [14]:
label1, label2

(tensor([781, 704, 145, 589, 604, 720, 586, 165, 493, 264, 497, 487, 188, 361,
         518,  22,  87, 196, 710, 572, 716, 643, 749, 433, 782, 787, 456, 479,
         387, 415, 727,  13, 471, 687,  20, 379, 200, 601,   2, 271, 730, 232,
         504, 694, 458, 649,  95, 513, 709, 269, 729, 769, 261, 435, 558, 652,
         202, 439, 548, 171,   3, 477,  89, 571]),
 tensor([287, 559, 755, 167, 732, 356, 325, 655, 474, 589, 397,  51, 264, 115,
         481, 389, 217,  57, 419, 727, 769, 300, 475, 194, 502, 535, 240, 736,
         323, 194, 316, 694, 729,  87, 121, 448, 533, 689, 107, 634, 691, 543,
          95, 302, 771, 770, 483, 429, 721,  38,   8, 717, 324,  56, 181, 658,
         363, 232, 712, 556, 494, 264, 232, 415]))

In [10]:
unbatched_eg = next(iter(loader))

## Test PL module

In [8]:
import lightning_scripts.lightning_ssl as lightning 
importlib.reload(lightning)

LitAudioSSL = lightning.LitAudioSSL
## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/pilot_ssl_audioset_resnet50_barlow.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 4 
# config['hparas']['batch_size'] = config['hparas']['global_batch_size'] / 1 # total batch size / n gpus 
config['hparas']['batch_size'] = 64 // 1 # total batch size / n gpus 

In [9]:
## Change loss functing 

# config['hparas']['ssl_task'] = 'dual'
# config['hparas']['ssl_loss_str'] = 'paired_mmcr'
# config['hparas']['ssl_loss'] = 'Paired_Loss'
# config['hparas']['ssl_loss_kwargs'] = dict(
#     loss_fn_inv='MMCR_Loss',
#     loss_fn_eq='MMCR_Loss',
#     loss_fn_eq_kwargs=None,
#     loss_fn_inv_kwargs=None,
#     lmda=0.4,
#     # out_dim=512,
#     # scale_loss = 0.04

# )


In [10]:
config['num_gpus'] = 1 

In [11]:
model = LitAudioSSL(config)

In [12]:
from lightning.pytorch.callbacks import ModelCheckpoint

from pathlib import Path

In [14]:
callbacks = []
# lr_monitor = LearningRateMonitor(logging_interval='step')
callbacks.append(lr_monitor)
checkpoint_dir = Path('exp') / Path(config_path).stem / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# if config.get('val_metric', None):
#     callbacks.append(ModelCheckpoint(
#                             checkpoint_dir,
#                             monitor=f"{config['val_metric']}",
#                             mode=config['val_metric_mode'],
#                             save_top_k=1,
#                             save_weights_only=True,
#                             verbose=True,
#                 ))
# callbacks.append(ModelCheckpoint(
#             checkpoint_dir,
#             monitor="train_total_loss",
#             mode="min",
#             save_top_k=1,
#             save_weights_only=True,
#             verbose=True,
#         ))

NameError: name 'lr_monitor' is not defined

In [17]:
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [18]:
trainer.fit(model)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name       | Type                       | Params | Mode 
------------------------------------------------------------------
0 | transforms | AudioCompose               | 0      | train
1 | audio_rep  | AudioToAudioRepresentation | 0      | train
2 | model      | ModelWithFrontEnd          | 24.8 M | train
3 | ssl_loss   | Barlow_Loss                | 0      | train
------------------------------------------------------------------
24.8 M    Trainable params
0         Non-trainable params
24.8 M    Total params
99.254    Total estimated model params size (MB)
169       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:298: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [7]:
from torchvision.models.resnet import resnet50
from robustness.audio_models import resnet50 as resnet50_robusntess
import torch.nn as nn 

class SSLAudioModelWMetamers(nn.Module):
    def __init__(self, projector_dims=[512, 512], proj_out_dim=2048, n_classes=794, supervised=False, **kwargs):
        super().__init__()
        self.supervised = supervised

        self.f = resnet50_robusntess()
        self.f.fc = nn.Identity()

        # projection head (Following exactly barlow twins offical repo)
        projector_dims = [proj_out_dim] + projector_dims
        layers = []
        for i in range(len(projector_dims) - 2):
            layers.append(
                nn.Linear(projector_dims[i], projector_dims[i + 1], bias=False)
            )
            layers.append(nn.BatchNorm1d(projector_dims[i + 1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(projector_dims[-2], projector_dims[-1], bias=False))
        self.g = nn.Sequential(*layers)
        if supervised:
            self.lin_cls = nn.Linear(proj_out_dim, n_classes)

    def forward(self, x):
        x_ = self.f(x)
        feature = torch.flatten(x_, start_dim=1)
        out = self.g(feature)
        if not self.supervised:
            return feature, out, None 
        else:
            logits = self.lin_cls(feature)
        return feature, out, logits

In [13]:
import torch

In [16]:
# model = SSLAudioModelWMetamers().cuda()

In [69]:
ckpt_path = "model_checkpoints/pilot_ssl_audioset_resnet50/checkpoints/epoch=9-step=37500-v1.ckpt"
# checkpoint = torch.load(ckpt_path, weights_only=True)
# # update state dict 
# new_state_dict = {}
# for key,val in checkpoint['state_dict'].items():
#     new_key = key.split('model.model.')[-1] if key.startswith("model.model.") else key 
#     new_state_dict[new_key] = val
# # model.load_state_dict(new_state_dict, strict=False)

In [56]:
# model.load_state_dict(checkpoint['state_dict'], strict=True)

<All keys matched successfully>

In [3]:
import lightning_scripts.lightning_ssl as lightning 
import importlib
import yaml
import torch

importlib.reload(lightning)

config_path = "model_configs/pilot_ssl_audioset_resnet50.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 4 
# config['hparas']['batch_size'] = config['hparas']['global_batch_size'] / 1 # total batch size / n gpus 
config['hparas']['batch_size'] = 64 // 1 # total batch size / n gpus 
ckpt_path = "model_checkpoints/pilot_ssl_audioset_resnet50/checkpoints/epoch=9-step=37500-v1.ckpt"

LitAudioSSL = lightning.LitAudioSSL
model = LitAudioSSL.load_from_checkpoint(checkpoint_path=ckpt_path, config=config).eval().cuda()

In [51]:
# model = model.cuda()

In [53]:
# f = model.model.model.f

In [4]:
model.model

ModelWithFrontEnd(
  (front_end): AudioToAudioRepresentation(
    (rep): AudioToCochleagram(
      (envelope_extraction): HilbertEnvelopeExtraction()
      (downsampling_op): SincWithKaiserWindow()
      (Cochleagram): Cochleagram(
        (compute_subbands): ComputeSubbands()
        (envelope_extraction): HilbertEnvelopeExtraction()
        (downsampling): SincWithKaiserWindow()
      )
    )
    (compression): ClippedGradPower(
      (compression_function): ClippedGradPowerCompression()
    )
  )
  (model): SSLAudioModelWMetamers(
    (f): ResNet(
      (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): SequentialWithArgs(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
   

In [6]:
x = torch.rand(1,1,40000).cuda()
# f(x, with_latent=True)
model.model(x, with_latent=True)

(tensor([[2.3782, 0.5847, 1.5378,  ..., 1.1603, 1.2008, 0.7588]],
        device='cuda:0', grad_fn=<ViewBackward0>),
 tensor([[2.3782, 0.5847, 1.5378,  ..., 1.1603, 1.2008, 0.7588]],
        device='cuda:0', grad_fn=<ViewBackward0>),
 {'input_after_preproc': tensor([[[[0.6600, 0.6591, 0.6593,  ..., 0.6582, 0.6619, 0.6693],
            [0.6594, 0.6598, 0.6596,  ..., 0.6589, 0.6597, 0.6689],
            [0.6588, 0.6606, 0.6599,  ..., 0.6594, 0.6578, 0.6689],
            ...,
            [0.2970, 0.2824, 0.2250,  ..., 0.2252, 0.2536, 0.2725],
            [0.2569, 0.2360, 0.1908,  ..., 0.1946, 0.1991, 0.2304],
            [0.1811, 0.1700, 0.1440,  ..., 0.1507, 0.1390, 0.1764]]]],
         device='cuda:0'),
  'conv1': tensor([[[[-0.1934, -0.4840, -0.5017,  ..., -0.4997, -0.5011, -0.5091],
            [ 0.0464, -0.2472, -0.2989,  ..., -0.3258, -0.3337, -0.3263],
            [-0.0434, -0.1679, -0.1895,  ..., -0.1962, -0.2206, -0.3024],
            ...,
            [-0.0856, -0.2461, -0.2175, 